🔹 STEP 2.1.1: Load Corpus

In [1]:
# evaluation/generate_questions.py

import json
import random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [27]:
TARGET_QUESTIONS = 1
MAX_TEXT_CHARS = 1500   # keep prompts manageable

INPUT_ARTICLES_FILE = "data/articles.json"
OUTPUT_QUESTIONS_FILE = "evaluation/questions.json"

# -----------------------------------------
# Load Wikipedia Articles
# -----------------------------------------
with open(INPUT_ARTICLES_FILE, "r", encoding="utf-8") as f:
    articles = json.load(f)

print(f"Loaded {len(articles)} articles")

Loaded 499 articles


🔹 STEP 2.1.2: Load LLM

In [22]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


🔹 STEP 2.1.3: Question Prompt Templates (IMPORTANT)

In [23]:
QUESTION_TEMPLATES = {
    "factual": (
        "From the text below, generate ONE factual question and its answer.\n\n"
        "Text:\n{text}\n\n"
        "STRICT FORMAT:\n"
        "Question: <question>\n"
        "Answer: <answer>"
    ),
    "inferential": (
        "From the text below, generate ONE inferential question that requires reasoning, "
        "and provide a short factual answer.\n\n"
        "Text:\n{text}\n\n"
        "STRICT FORMAT:\n"
        "Question: <question>\n"
        "Answer: <answer>"
    ),
    "comparative": (
        "From the text below, generate ONE comparative question comparing two concepts "
        "mentioned in the text, and provide its answer.\n\n"
        "Text:\n{text}\n\n"
        "STRICT FORMAT:\n"
        "Question: <question>\n"
        "Answer: <answer>"
    ),
    "multi-hop": (
        "From the text below, generate ONE multi-hop question that requires combining "
        "at least two facts from the text, and provide its answer.\n\n"
        "Text:\n{text}\n\n"
        "STRICT FORMAT:\n"
        "Question: <question>\n"
        "Answer: <answer>"
    )
}


In [24]:
QUESTION_TYPES = list(QUESTION_TEMPLATES.keys())

🔹 STEP 2.1.4: Generate a Single Q&A Pair

In [18]:
# -----------------------------------------
# Robust Q&A Parser
# -----------------------------------------
def parse_qa(decoded_text):
    decoded_text = decoded_text.strip()

    # Case 1: Strict format
    if "Question:" in decoded_text and "Answer:" in decoded_text:
        question = decoded_text.split("Question:")[1].split("Answer:")[0].strip()
        answer = decoded_text.split("Answer:")[1].strip()
        return question, answer

    # Case 2: Q:/A: format
    if "Q:" in decoded_text and "A:" in decoded_text:
        question = decoded_text.split("Q:")[1].split("A:")[0].strip()
        answer = decoded_text.split("A:")[1].strip()
        return question, answer

    return None, None

In [19]:
# -----------------------------------------
# Generate Q&A from One Article
# -----------------------------------------
def generate_qa(article_text, question_type):
    prompt = QUESTION_TEMPLATES[question_type].format(
        text=article_text[:MAX_TEXT_CHARS]
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        do_sample=False
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    question, answer = parse_qa(decoded)

    # Basic quality filters
    if question is None or answer is None:
        return None, None

    if len(question.split()) < 5 or len(answer.split()) < 3:
        return None, None

    return question, answer

🔹 STEP 2.1.5: Generate 100 Questions (CORE LOOP)

In [29]:
# -----------------------------------------
# Main Generation Loop
# -----------------------------------------
generated_questions = []
question_id = 1

random.shuffle(articles)

for article in articles:
    if len(generated_questions) >= TARGET_QUESTIONS:
        break

    q_type = random.choice(QUESTION_TYPES)

    print(f"Generating {q_type} question for article: {article['title']}")

    question, answer = generate_qa(article["text"], q_type)
    print(f"Generated Question: {question}")

    if question is None:
        print("⚠️ Skipped (invalid format)")
        continue

    generated_questions.append({
        "id": question_id,
        "question": question,
        "ground_truth_answer": answer,
        "source_url": article["url"],
        "source_title": article["title"],
        "question_type": q_type
    })

    print("✔ Question generated")
    question_id += 1

# -----------------------------------------
# Save Generated Questions
# -----------------------------------------
with open(OUTPUT_QUESTIONS_FILE, "w", encoding="utf-8") as f:
    json.dump(generated_questions, f, indent=2)

print(f"\n✅ Saved {len(generated_questions)} questions to {OUTPUT_QUESTIONS_FILE}")

Generating comparative question for article: Yubileyny Sports Palace
Generated Question: None
⚠️ Skipped (invalid format)
Generating multi-hop question for article: Ernst Engelberg
Generated Question: None
⚠️ Skipped (invalid format)
Generating comparative question for article: Artificial Intelligence: A Modern Approach
Generated Question: None
⚠️ Skipped (invalid format)
Generating inferential question for article: Alfred Grévin
Generated Question: None
⚠️ Skipped (invalid format)
Generating inferential question for article: Nikolay Kovalyov (politician)
Generated Question: None
⚠️ Skipped (invalid format)
Generating multi-hop question for article: Salon of 1841


KeyboardInterrupt: 